# «Узнать товар»: дообучение поиска по виду

Датасет «uznat-tovar-photos» (фото товаров магазина, `ml/shop_dataset.py`) должен быть подключён (Add Input).
Нужны: **Accelerator — GPU T4**, **Internet — On** (модель SigLIP 2 скачивается с HuggingFace, ~375 МБ — это делает Kaggle, не ваш интернет).
Время: около 30–40 минут. Результат — во вкладке Output: `lora_final.pt` (~5 МБ), `cv_*.npy`, `base.npy`.

In [ ]:
import glob, os, shutil, zipfile
csv = glob.glob('/kaggle/input/**/photos.csv', recursive=True)
assert csv, 'Датасет с photos.csv не подключён: Add Input -> Your Datasets -> uznat-tovar-photos'
SRC = os.path.dirname(csv[0])
DATA = SRC
if not os.path.isdir(os.path.join(SRC, 'images')):          # Kaggle мог оставить папку фото архивом
    DATA = '/kaggle/temp/data'
    os.makedirs(DATA, exist_ok=True)
    for name in ('photos.csv', 'finetune_embedder.py'):
        shutil.copy(os.path.join(SRC, name), DATA)
    for z in glob.glob(os.path.join(SRC, '**', '*.zip'), recursive=True):
        zipfile.ZipFile(z).extractall(DATA)
print('Данные:', DATA, '| фото:', len(glob.glob(os.path.join(DATA, 'images', '*.jpg'))))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python {DATA}/finetune_embedder.py --data {DATA} --out /kaggle/working --cv --final